# Student Success Copilot

**AI-powered study planning assistant with early risk detection**

This notebook demonstrates a hybrid AI system that combines:
1. **Search-based planner** — A\* and Greedy algorithms for schedule optimisation
2. **Rule-based expert system** — forward and backward chaining with confidence scores
3. **ML classifier** — RandomForest for risk prediction with evaluation metrics
4. **Generative AI tutor** — Anthropic Claude API with guardrails and prompting strategy

---
**How to run:** Execute cells top-to-bottom. Sections 3c and 6 will pause for your input — the system asks a question and you type your answer.

---
## 0. Project Setup

The cell below clones the project from GitHub into the Colab environment and adds it to the Python path.

In [ ]:
# ── Reset working directory ──────────────────────────────────────────
%cd /content

# ── Clean broken folder (if exists) ─────────────────────────────────
import shutil, os
if os.path.exists('/content/wiut_ai_cw_level4'):
    shutil.rmtree('/content/wiut_ai_cw_level4')

# ── Clone project ───────────────────────────────────────────────────
!git clone https://github.com/akramovvv/wiut_ai_cw_level4.git

# ── Setup path ──────────────────────────────────────────────────────
import sys, warnings
warnings.filterwarnings('ignore')

PROJECT_PATH = '/content/wiut_ai_cw_level4'

sys.path.insert(0, PROJECT_PATH)
os.chdir(PROJECT_PATH)

print("Project loaded from GitHub")
print("Working directory:", os.getcwd())
print("Contents:", os.listdir(PROJECT_PATH))

---
## 1. Synthetic Dataset Generation

500 student profiles, 10 features, 3 risk labels (Low / Medium / High).
~12% controlled noise prevents the ML model from perfectly memorising the rules.

In [ ]:
from data.generate_dataset import main as generate_dataset
generate_dataset()

---
## 2. ML Model — Training & Evaluation

RandomForestClassifier · 80/20 stratified split · Accuracy + F1 + confusion matrix

In [ ]:
from ml.train import main as train_ml
train_ml()

---
## 3. Rule-Based Expert System

### 3a. Knowledge Base — 11 rules with severity and confidence scores

In [ ]:
from rules.knowledge_base import RULES

print(f"  {'ID':<30} {'Severity':<10} {'Conf':<8} Requires")
print("  " + "-" * 62)
for r in RULES:
    print(f"  {r['id']:<30} {r['severity']:<10} {r['confidence']:.0%}     {r['requires']}")

### 3b. Forward Chaining (data-driven)

Fires all matching rules from facts → derives risk level.

In [ ]:
from rules.forward_chain import run_forward_chain

profile_normal = {
    "free_hours_per_day": 5.0, "num_pending_tasks": 3.0,
    "avg_task_duration_hours": 2.0, "deadline_urgency": 0.2,
    "days_until_exam": 14.0, "stress_level": 4.0,
    "sleep_hours_avg": 7.5, "past_completion_rate": 0.85,
    "extracurricular_hours": 6.0, "missed_classes_pct": 0.05,
}
profile_atrisk = {
    "free_hours_per_day": 2.0, "num_pending_tasks": 11.0,
    "avg_task_duration_hours": 4.0, "deadline_urgency": 0.9,
    "days_until_exam": 2.0, "stress_level": 8.0,
    "sleep_hours_avg": 4.5, "past_completion_rate": 0.3,
    "extracurricular_hours": 15.0, "missed_classes_pct": 0.4,
}

for label, profile in [("Normal", profile_normal), ("At-risk", profile_atrisk)]:
    fc = run_forward_chain(profile)
    print(f"[{label}]  risk={fc.risk_level}  rules fired={len(fc.fired_rules)}  signals={fc.signals}")
    print(f"         {fc.explanation[:120]}...")
    print()

### 3c. Backward Chaining (goal-driven) — Interactive Demo

`stress_level` is intentionally missing from the profile below.
The system detects this, asks **you** for a value, and then shows how the conclusion changes.

**Try different values** (1–10) to see how risk level shifts.

In [ ]:
from rules.backward_chain import run_backward_chain
from rules.forward_chain import run_forward_chain

# stress_level is intentionally missing — backward chain will detect and ask
profile_incomplete = {
    "free_hours_per_day": 2.0, "num_pending_tasks": 11.0,
    "avg_task_duration_hours": 4.0, "deadline_urgency": 0.9,
    "sleep_hours_avg": 4.5, "past_completion_rate": 0.3,
    "extracurricular_hours": 15.0, "missed_classes_pct": 0.4,
    "days_until_exam": 2.0,
}

# ── BEFORE: analyse without stress_level ─────────────────────────────
fc_before = run_forward_chain(profile_incomplete)
print("BEFORE (stress_level missing):")
print(f"  Risk level : {fc_before.risk_level}")
print(f"  Rules fired: {len(fc_before.fired_rules)}")
print(f"  Signals    : {fc_before.signals}")
print()

# ── INTERACTIVE: backward chain asks the user ────────────────────────
def interactive_callback(key, question_text):
    """Asks the user for input via input() — works in Colab."""
    print(question_text.strip())
    return input(">>> Your answer: ")

bc = run_backward_chain(profile_incomplete, io_callback=interactive_callback)
print(f"\n  Questions asked : {bc.questions_asked}")
print(f"  Answers received: {bc.answers_received}")

# ── AFTER: re-analyse with the user's answer ─────────────────────────
fc_after = run_forward_chain(bc.profile_complete)
print(f"\nAFTER (stress_level = {bc.profile_complete.get('stress_level')}):")
print(f"  Risk level : {fc_after.risk_level}")
print(f"  Rules fired: {len(fc_after.fired_rules)}")
print(f"  Signals    : {fc_after.signals}")
print(f"\n  The system improved its conclusion using your input.")

---
## 4. Search-Based Planner — A\* vs Greedy

| | A\* | Greedy |
|---|---|---|
| Strategy | Best-first, f = g + h | Earliest deadline first |
| Optimality | Guaranteed (admissible h) | Not guaranteed |
| Speed | Slower (more nodes) | Faster (O n log n) |

In [ ]:
from planner.astar  import astar_schedule, compare_algorithms
from planner.greedy import greedy_schedule
from planner.state  import make_time_slots, make_tasks_from_profile

# ── Normal: 4 tasks, 14 slots ─────────────────────────────────
tasks_normal = [
    {"name": "Math Essay", "duration": 1, "deadline": 3},
    {"name": "ML Lab",     "duration": 1, "deadline": 5},
    {"name": "OS Report",  "duration": 1, "deadline": 9},
    {"name": "DB Project", "duration": 1, "deadline": 12},
]
slots_normal = make_time_slots(free_hours_per_day=4, days=7)

astar_n  = astar_schedule(tasks_normal, slots_normal)
greedy_n = greedy_schedule(tasks_normal, slots_normal)
print(compare_algorithms(astar_n, greedy_n, tasks_normal, slots_normal))

# ── At-risk: 11 tasks, 7 slots (overflow) ─────────────────────
tasks_risk = make_tasks_from_profile({"num_pending_tasks": 11, "days_until_exam": 3,
                                      "avg_task_duration_hours": 2, "free_hours_per_day": 2})
slots_risk = make_time_slots(free_hours_per_day=2, days=7)

astar_r  = astar_schedule(tasks_risk, slots_risk)
greedy_r = greedy_schedule(tasks_risk, slots_risk)
print(compare_algorithms(astar_r, greedy_r, tasks_risk, slots_risk))

---
## 5. Full Pipeline — Scenario A (Normal Student)

`run_copilot()` runs all components in sequence:
backward chain → ML predict → forward chain → A\* planner → explanation card

In [ ]:
from copilot import run_copilot

profile_a = {
    "free_hours_per_day": 5,  "num_pending_tasks": 4,
    "avg_task_duration_hours": 2, "deadline_urgency": 0.2,
    "days_until_exam": 14,    "stress_level": 4,
    "sleep_hours_avg": 7.5,   "past_completion_rate": 0.85,
    "extracurricular_hours": 6, "missed_classes_pct": 0.05,
}

result_a = run_copilot(profile_a)
result_a.print_report()

---
## 6. Full Pipeline — Scenario B (At-Risk Student)

`stress_level` is missing → backward chain asks **you** for it → the full pipeline runs with your answer.

In [ ]:
from copilot import run_copilot

profile_b = {
    "free_hours_per_day": 2,  "num_pending_tasks": 11,
    "avg_task_duration_hours": 2, "deadline_urgency": 0.9,
    "days_until_exam": 2,
    # stress_level missing on purpose — system will ask you
    "sleep_hours_avg": 4.5,   "past_completion_rate": 0.3,
    "extracurricular_hours": 15, "missed_classes_pct": 0.4,
}

def interactive_student(key, question_text):
    """Asks the user for input via input() — works in Colab."""
    print(question_text.strip())
    return input(">>> Your answer: ")

result_b = run_copilot(profile_b, io_callback=interactive_student)
result_b.print_report()

---
## 7. Generative AI Component

### 7a. Guardrails — input and output safety filters

In [ ]:
from genai.guardrails import validate_input, validate_output

cases = [
    ("Schedule my study sessions",   "normal"),
    ("I need medical diagnosis",     "blocked topic"),
    ("Write my essay for me",        "off-topic"),
    ("You should see a doctor now.", "unsafe output"),
]

for text, label in cases[:3]:
    ok, reason = validate_input(text)
    print(f"  INPUT  [{('PASS' if ok else 'BLOCK'):>5}]  {label:<16} → {reason}")

ok, reason = validate_output(cases[3][0])
print(f"  OUTPUT [{'PASS' if ok else 'BLOCK':>5}]  {cases[3][1]:<16} → {reason}")

### 7b. GenAI Tutor — personalised tips from result_b

Set `ANTHROPIC_API_KEY` to get live Claude responses. Without it, safe fallback text is shown.

In [ ]:
import os
from genai.tutor import generate_study_tips, explain_algorithms, coach_at_risk

print("Study tips:")
print(generate_study_tips(result_b))
print("\nAlgorithm explanation:")
print(explain_algorithms(result_b))
print("\nRisk coach:")
print(coach_at_risk(result_b))

---
## 8. Limitations & Responsible AI

In [ ]:
from ml.train import ML_LIMITATIONS
from genai.guardrails import GENAI_RISKS

print(ML_LIMITATIONS)
print()
print(GENAI_RISKS)

---
## 9. Summary

| Component | Requirement | Implementation |
|-----------|-------------|----------------|
| **Search planner** | >= 2 strategies, compare | A\* + Greedy, comparison table |
| **Rule-based system** | Forward + backward chaining, confidence | 11 rules, confidence 0.55-0.90, both chaining types |
| **ML model** | Train/test split, metrics | RF classifier, 80/20 split, accuracy + F1 + confusion matrix |
| **GenAI** | Guardrails, prompting strategy, risks | 3 modes, input/output validators, fallback, risk paragraph |
| **Interactive loop** | Ask when data missing | Backward chaining asks for `stress_level` |
| **Outputs** | Schedule + risk + explanation | Weekly plan, Low/Med/High risk, explanation card |

---
*Student Success Copilot — AI Coursework Project*